<a href="https://colab.research.google.com/github/GAURAV4478/cookbook/blob/sql-agent-notebook/examples/langchain/SQL_Agent_Gemini_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [78]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: SQL Agent using LangChain and SQLite

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Template.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/GAURAV4478" target="_blank" title="View Gaurav's profile on GitHub">
        <img src="https://github.com/GAURAV4478.png?size=100"
             alt="GAURAV4478's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/GAURAV4478" target="_blank"><font color='#217bfe'><strong>Gaurav Thakur</strong></font></a>.</font></h2>
      <h5><font color='black'>
        <a href="https://linkedin.com/in/gauravthakur7" target="_blank"><font color="#078efb">LinkedIn</font></a><br>
        <a href="https://github.com/GAURAV4478" target="_blank"><font color="#078efb">GitHub</font></a>
      </h5></font><br>
  </tr>
</table>

This notebook demonstrates how to build an autonomous SQL Agent
using the Gemini API and LangChain. Unlike a traditional chain-based
approach, a SQL Agent can reason about your question, decide which
queries to run, and self-correct errors — all on its own.

By the end of this notebook, you will be able to:
- Set up a SQLite database with sample data
- Create a LangChain SQL Agent powered by Gemini
- Query the database using plain natural language

## Setup

### Install SDK

In [79]:
%pip install -U -q "google-genai>=2.9.0" langchain langchain-community langchain-google-genai

In [80]:
import sqlite3
import pandas as pd
from sklearn.datasets import fetch_openml

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import Markdown

### Set up your API key

You'll need a Gemini API key to run this notebook.

1. Get your free API key from [Google AI Studio](https://aistudio.google.com/apikey)
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar
3. Add a new secret with name `GEMINI_API_KEY` and paste your key as the value
4. Enable notebook access for the secret

In [81]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [82]:
MODEL_ID = "gemini-3.1-flash-lite" # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

# Ideally order the model by "cabability" ie. generation then within generation
# 8b/flash-lite then flash then pro

> **Note:** If you encounter a `429 RESOURCE_EXHAUSTED` error, your free tier quota for the selected model may be exhausted. Try switching to a different model in the model selection cell above and re-run the notebook.

## Setting up the database

In this section, you will create a SQLite database using the Titanic dataset.
The dataset contains information about passengers including age, gender, ticket class, fare, and survival status.


1. To query a database, you first need to set one up. Here you will use the Titanic dataset loaded directly from OpenML.

In [83]:
titanic = fetch_openml(name="titanic", version=1, as_frame=True)
df = titanic.frame[["pclass", "survived", "name", "sex", "age", "fare", "embarked"]].copy()

2. **Clean the data:** Convert columns to correct data types and remove rows with missing values.

In [84]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["survived"] = df["survived"].astype(int)
df["pclass"] = df["pclass"].astype(int)
df.dropna(inplace=True)

3. **Create the SQLite database:** Store the cleaned data in a SQLite database called `titanic.db` inside a table named `passengers`.

In [85]:
conn = sqlite3.connect("titanic.db")
df.to_sql("passengers", conn, index=False, if_exists="replace")

1043

## Create the SQL agent

Now that the database is ready, create a SQL agent using LangChain and Gemini.

In [86]:
# Initialize Gemini as the LLM
llm = ChatGoogleGenerativeAI(
    model=MODEL_ID,
    google_api_key=GEMINI_API_KEY
)

Create a SQLDatabase object to connect to the Titanic database.

In [87]:
db = SQLDatabase.from_uri("sqlite:///titanic.db")
print(db.get_table_info())


CREATE TABLE passengers (
	pclass INTEGER, 
	survived INTEGER, 
	name TEXT, 
	sex TEXT, 
	age REAL, 
	fare REAL, 
	embarked TEXT
)

/*
3 rows from passengers table:
pclass	survived	name	sex	age	fare	embarked
1	1	Allen, Miss. Elisabeth Walton	female	29.0	211.3375	S
1	1	Allison, Master. Hudson Trevor	male	0.9167	151.55	S
1	0	Allison, Miss. Helen Loraine	female	2.0	151.55	S
*/


Create the SQL agent by combining the LLM and the database.

In [88]:
# Create the SQL agent
agent = create_sql_agent(
    llm=llm,
    db=db,
    verbose=False,
    agent_type="openai-tools",
    handle_parsing_errors=True,
    prefix="""You are a helpful data analyst assistant.
    When answering questions, always explain your findings in clear, simple English.
    Always provide the exact numbers from the database in your response."""
)

> **Note:** To see the agent's internal reasoning and generated SQL queries, set `verbose=True` in the agent creation cell above.

## Query the database

Now that the agent is ready, query the database using plain English. The agent will automatically generate the SQL query, execute it, and return the answer.

Before querying, define a helper function to display the question, the SQL query generated by the agent, and the final answer in a clean format.

In [89]:
# Helper function to run and display agent queries
def ask_agent(question):
    response = agent.invoke({"input": question})

    print(f"Question: {question}")
    print(f"Answer: {response['output'][0]['text']}")
    print("-" * 50)

## Example queries

The following examples demonstrate the agent's ability to answer questions of varying complexity — from simple counts to multi-condition analytical queries.

In [90]:
ask_agent("How many passengers survived and how many did not?")

Question: How many passengers survived and how many did not?
Answer: Based on the data, 425 passengers survived, and 618 passengers did not survive.
--------------------------------------------------


In [91]:
ask_agent("What was the survival rate percentage for each combination of gender and passenger class?")

Question: What was the survival rate percentage for each combination of gender and passenger class?
Answer: The survival rates for passengers, broken down by gender and passenger class, are as follows:

*   **Females:**
    *   **1st Class:** 96.18%
    *   **2nd Class:** 89.32%
    *   **3rd Class:** 47.37%

*   **Males:**
    *   **1st Class:** 35.10%
    *   **2nd Class:** 14.56%
    *   **3rd Class:** 16.95%

In general, females had a significantly higher survival rate than males across all passenger classes, and survival rates decreased as the passenger class went from 1st to 3rd (with the exception of 3rd-class males, who had a slightly higher survival rate than 2nd-class males).
--------------------------------------------------


In [92]:
ask_agent("Which age group had the highest survival rate? Group ages into child (0-12), teenager (13-17), adult (18-60), and senior (60+).")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite\nPlease retry in 49.971377492s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.1-

In [ ]:
ask_agent("What is the ratio of survivors to non survivors for each passenger class, and which class had the best odds of survival?")

## Summary

Congratulations! You have successfully built a SQL Agent that can answer natural language questions about the Titanic dataset. Feel free to explore further by asking your own questions about the data.

## Next steps

In this notebook, you built a SQL Agent using Gemini and LangChain that can answer natural language questions about a database.

To learn more, check out the following resources:

- [Gemini API documentation](https://ai.google.dev/gemini-api/docs)
- [LangChain SQL Agent documentation](https://python.langchain.com/docs/tutorials/sql_qa/)
- [LangChain Google Generative AI](https://python.langchain.com/docs/integrations/chat/google_generative_ai/)
- [SQLite documentation](https://www.sqlite.org/docs.html)

### Related examples
Browse more examples in the [Gemini Cookbook](https://github.com/google-gemini/cookbook/tree/main/examples/langchain).